# 🎵 Music Recommendation System

A simple music recommendation system built using Spotify data.

The goal of this project is to recommend songs that are similar to a song the user selects, based on its audio features, genre, and other information.

## How It Works

The basic idea is pretty simple:

1. Load and clean the Spotify dataset.
2. Select the features that are useful for finding similar songs.
3. Scale the numerical features and encode the genres.
4. Combine everything into one feature matrix.
5. Find songs that are most similar using cosine similarity.
6. Use genre and artist information to improve the recommendations.
7. Return the most similar songs to the user.



# 🎵 Music Recommendation System

This project builds a content-based music recommendation system using Spotify track metadata and audio features.

The system recommends songs based on their similarity in musical characteristics such as danceability, energy, acousticness, valence, and genre.

In [ ]:
import pandas as pd
import numpy as np


## 1. Load the Dataset

First, let's load the Spotify dataset and take a look at what we're working with.



In [ ]:
songs = pd.read_csv('spotify-tracks-dataset-detailed.csv')
songs.head()


## 2. Data Selection & Cleaning

There are quite a few columns in the original dataset, so I'll keep the ones that are useful for the recommendation system and remove missing values.



In [ ]:
songs = songs[[
    'track_name',
    'artists',
    'track_genre',
    'danceability',
    'popularity',
    'energy',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence'
]]

songs = songs.dropna().drop_duplicates().reset_index(drop=True)

print('Rows:', len(songs))
print('Columns:', len(songs.columns))


## 3. Remove Duplicate Tracks

The dataset contains duplicate tracks, so I'll remove them to avoid getting the same song multiple times in the recommendations.



In [ ]:
songs = songs.drop_duplicates(
    subset=['track_name', 'artists']
).reset_index(drop=True)

print('Unique songs:', len(songs))


## 4. Normalize Song Titles

I'll clean up the song titles a little by converting them to lowercase and removing the extra text inside parentheses.



In [ ]:
songs['clean_title'] = (
    songs['track_name']
    .str.lower()
    .str.split('(')
    .str[0]
    .str.strip()
)


## 5. Select Audio Features

Now I'll select the audio features that will be used to compare songs with each other.



In [ ]:
numerical_features = [
    'danceability',
    'popularity',
    'energy',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence'
]

X_numeric = songs[numerical_features]


## 6. Feature Scaling

The audio features have different ranges, so I'll scale them before using them for similarity calculations.



In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric)


## 7. Encode Music Genres

Since genre is a categorical feature, I'll convert it into numerical values using one-hot encoding.



In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore')
X_genre = encoder.fit_transform(songs[['track_genre']])


## 8. Build the Recommendation Feature Matrix

Now I'll combine the scaled audio features and encoded genres into one feature matrix. This will be the data used to find similar songs.



In [ ]:
from scipy.sparse import hstack, csr_matrix

X = hstack([
    csr_matrix(X_numeric_scaled),
    X_genre * 0.3
])

print('Songs:', len(songs))
print('Feature matrix:', X.shape)


## 9. Recommendation Engine

Now comes the main part of the project.

I'll use cosine similarity to find songs that are closest to the selected song based on their features.



In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_song(song_name, n=10):

    # Find the song
    matches = songs[
        songs['track_name'].str.lower() == song_name.lower()
    ]

    if matches.empty:
        return "Song not found."

    index = matches.index[0]

    # Input song details
    input_genre = songs.loc[index, 'track_genre']
    input_title = songs.loc[index, 'clean_title']
    input_artists = str(
        songs.loc[index, 'artists']
    ).split(';')

    # Similarity
    similarity = cosine_similarity(
        X[index], X
    ).flatten()

    results = songs.copy()
    results['similarity'] = similarity

    # Genre match
    results['genre_match'] = (
        results['track_genre'] == input_genre
    ).astype(float)

    # Artist match
    def same_artist(artists):
        if pd.isna(artists):
            return False

        return any(
            artist in str(artists).split(';')
            for artist in input_artists
        )

    results['artist_match'] = (
        results['artists']
        .apply(same_artist)
        .astype(float)
    )

    # Final score
    results['score'] = (
        results['similarity'] * 0.85
        + results['genre_match'] * 0.10
        + results['artist_match'] * 0.05
    )

    # Remove original song
    results = results[
        results.index != index
    ]

    # Remove same song / alternate versions
    results = results[
        results['clean_title'] != input_title
    ]

    # Sort
    results = results.sort_values(
        'score',
        ascending=False
    )

    # Remove duplicate versions
    results = results.drop_duplicates(
        subset='clean_title'
    )

    # Return clean result
    return results.head(n)[[
        'track_name',
        'artists',
        'track_genre',
        'score'
    ]].reset_index(drop=True)
  

In [ ]:
recommend_song('Tum Hi Ho', 10)


In [ ]:
recommend_song('Good to Me', 10)


## 10. Test Recommendations

Let's try the recommendation system with a few different songs and see what it comes up with.



In [ ]:

recommend_song("I'm Yours", 10)

## 11. Save Processed Data

The preprocessing takes some time, so I'll save the processed data and feature matrix as pickle files.

The Streamlit app can load these files directly instead of doing all the preprocessing again.



In [ ]:
import pickle

pickle.dump(songs, open('songs.pkl', 'wb'))
pickle.dump(X, open('X.pkl', 'wb'))

In [ ]:
!pip install spotipy

In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from dotenv import load_dotenv
import os

load_dotenv()

client_id = os.getenv("SPOTIFY_CLIENT_ID")
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")


sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(
        client_id=client_id,
        client_secret=client_secret
    )
)

print("Spotify connected!")

In [ ]:
import requests

In [ ]:
def get_cover_url(track_name, artist):
    try:
        url = "https://itunes.apple.com/search"

        params = {
            "term": f"{track_name} {artist}",
            "media": "music",
            "entity": "song",
            "limit": 5
        }

        response = requests.get(url, params=params, timeout=10)

        if response.status_code != 200:
            return None

        results = response.json()["results"]

        if len(results) == 0:
            return None

        # Take the first result
        cover_url = results[0].get("artworkUrl100")

        if cover_url:
            # Get a larger image
            cover_url = cover_url.replace("100x100", "600x600")

        return cover_url

    except Exception as e:
        print(f"Error for {track_name} - {artist}: {e}")
        return None

In [ ]:
cover = get_cover_url("Rait Zara Si", "A.R. Rahman")

print(cover)

In [ ]:
print(type(songs))

In [ ]:
songs.head()

In [ ]:
from IPython.display import Image, display

cover = get_cover_url("Believer", "Imagine Dragons")

print("Cover URL:", cover)

if cover:
    display(Image(url=cover))
else:
    print("No cover found")

In [ ]:
song = "Rait Zara Si"
artist = "A.R. Rahman"

cover = get_cover_url(song, artist)

if cover:
    display(Image(url=cover))
else:
    print("No cover found")

In [ ]:
result = recommend_song("Rait Zara Si", 10)
print(result)

In [ ]:
recommend_song("Rait Zara Si", 10)

In [ ]:
from IPython.display import Image, display

for _, row in result.iterrows():
    print(f"{row['track_name']} — {row['artists']}")
    
    cover_url = get_cover_url(row['track_name'], row['artists'])
    
    if cover_url:
        display(Image(url=cover_url, width=150))
    else:
        print("Cover not found")
    
    print("-" * 50)

## Conclusion

The recommendation system is now able to take a song and find other songs that are similar to it using Spotify's audio features and genre information.

The processed files can then be used by the Streamlit app to generate recommendations.
